# Loading Data and Defining Models

In [ ]:
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
import numpy as np
import matplotlib.pyplot as plt
plt.rcParams.update({
    "font.family": "serif",
    "font.serif": ["Liberation Serif"], # Matches Times New Roman
    "font.size": 10,                    # Standard for most papers
    "axes.labelsize": 10,
    "xtick.labelsize": 8,               
    "ytick.labelsize": 8,
    "legend.fontsize": 9,
    "text.usetex": False,               
    "pdf.fonttype": 42,               
    "ps.fonttype": 42
})
df_deep_agg = pd.read_csv("/kaggle/input/deep-features/deep_agg_features.csv")
df_handcrafted = pd.read_csv("/kaggle/input/handcrafted-features/Handcrafted_Features_flair.csv")
df_fused = pd.merge(df_deep_agg, df_handcrafted, on='id', how='inner')


X = df_fused.drop(columns=['id', 'label'])   # fused deep + radiomic features
y = df_fused['label']                       
print(f"Fused y: {np.bincount(y)}")             

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# SHAP VISUALISATION

In [ ]:
import shap
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

print("Generating BraTS SHAP Visualizations...")

class_names = ["LGG", "HGG"]

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)

feature_names = list(X.columns)

lr = LogisticRegression(max_iter=10000, solver='lbfgs', C=1.0, random_state=42, n_jobs=-1)
lr.fit(X_train_scaled, y_train)

# 3. Compute SHAP Values
explainer = shap.LinearExplainer(lr, X_train_scaled, feature_dependence="independent")
shap_values = explainer.shap_values(X_test_scaled)

X_test_df = pd.DataFrame(X_test_scaled, columns=feature_names)

# Bar Plot
plt.figure()
shap.summary_plot(
    shap_values, 
    X_test_df, 
    plot_type="bar", 
    class_names=class_names,
    max_display=20,
    show=True
)

# Dot Plot 
plt.figure()
# For binary classification with LinearExplainer, shap_values is a single matrix or 2-element list
sv_plot = shap_values[1] if isinstance(shap_values, list) else shap_values
shap.summary_plot(
    sv_plot, 
    X_test_df, 
    max_display=20,
    show=True
)
is_deep = np.array(["deep" in col.lower() or "resnet" in col.lower() or i < 2048 for i, col in enumerate(feature_names)])
is_radiomic = ~is_deep

abs_sv = np.abs(sv_plot).mean(axis=0)
deep_share = abs_sv[is_deep].sum() / abs_sv.sum() * 100.0
rad_share  = abs_sv[is_radiomic].sum() / abs_sv.sum() * 100.0

print("\n BRATS MODALITY ATTRIBUTION SHARE")
print(f"Deep Feature Share       : {deep_share:.2f}%")
print(f"Handcrafted Radiomic Share: {rad_share:.2f}%")